In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по свиньям v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Свиньи
404,АТЫРАУСКАЯ ОБЛАСТЬ,2019-08-01,5.40
1496,ОБЛАСТЬ ЖЕТІСУ,2023-12-01,242.23
1679,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2015-05-01,1437.63
744,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2015-11-01,310.92
668,ГШЫМКЕНТ,2020-02-01,1.90
403,АТЫРАУСКАЯ ОБЛАСТЬ,2019-05-01,0.20
1458,ОБЛАСТЬ АБАЙ,2023-12-01,156.41
203,АКТЮБИНСКАЯ ОБЛАСТЬ,2021-05-01,821.30
1506,ОБЛАСТЬ ЖЕТІСУ,2024-10-01,269.10
560,ГАЛМАТЫ,2016-05-01,0.54


In [3]:
regions = df['Регион'].unique()
target   = "Свиньи"
horizon  = 3
epsilon = 1e-6

In [4]:
first_test = pd.to_datetime("2024-08-01")
last_possible = df["Период"].max() - pd.DateOffset(months=horizon-1)
test_starts = pd.date_range(first_test, last_possible, freq="MS")

## Holt-Winter's (log)

In [5]:
results_hw = []
for region in regions:
    ts = (df[df["Регион"] == region]
          .set_index("Период")[target]
          .dropna()
          .sort_index())
    if len(ts) < 24:
        print(f"{region}: всего {len(ts)} мес. — сезонный Holt-Winter's невозможен.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        # формируем train / test
        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]

        # пропускаем, если недостаточно данных или неполный test
        if len(train) < 24 or len(test) < horizon:
            continue

        # обучаем модель
        train_log = np.log1p(train)

        hw_log = ExponentialSmoothing(
            train_log,
            seasonal="add",
            seasonal_periods=12
        ).fit(optimized=True)

        # прогноз и метрики
        fc_log = hw_log.forecast(horizon)
        fc = np.expm1(fc_log) 
        # fc   = model.forecast(horizon)
        rmse = np.sqrt(mean_squared_error(test, fc))
        mae  = mean_absolute_error(test, fc)
        mape = (np.abs((test - fc) / test).mean()) * 100

        results_hw.append({
            "Регион":      region,
            "Test start":  test_start.strftime("%Y-%m"),
            "Test end":    test_end.strftime("%Y-%m"),
            "Forecast":    [x.round(2) for x in list(fc.values)],
            "Actual":      [y.round(2) for y in list(test.values)],
            "RMSE":        rmse,
            "MAE":         mae,
            "MAPE_%":      mape
        })

# 4) Усреднение по всем скользящим окнам для каждого региона
res_hw = pd.DataFrame(results_hw)
res_hw.to_excel("results/Свиньи - Результаты прогнозов ХВ v2.xlsx", index=False)
print("Результаты прогнозов HW на 3 месяца:")

display(res_hw)

final_hw = (
    res_hw
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_hw.to_excel("results/Свиньи - Результаты прогнозов ХВ средние v2.xlsx", index=False)
print("Средние метрики Holt–Winter's по регионам (rolling-3):")
display(final_hw)

Результаты прогнозов HW на 3 месяца:


,Регион,Test start,Test end,Forecast,Actual,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"[452.55, 535.04, 559.53]","[553.23, 566.96, 496.85]",70.91,65.09,12.15
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"[579.8, 605.93, 796.64]","[566.96, 496.85, 488.59]",188.82,143.33,29.09
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"[600.71, 789.75, 1297.47]","[496.85, 488.59, 480.05]",506.51,407.48,84.27
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"[734.57, 1207.56, 319.13]","[488.59, 480.05, 263.72]",444.54,342.97,74.30
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"[1010.19, 267.49, 343.74]","[480.05, 263.72, 340.79]",306.09,178.95,37.58
...,...,...,...,...,...,...,...,...
148,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"[5.01, 0.94, 2.01]","[0.4, 1.1, 1.8]",2.66,1.66,NaN
149,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"[-0.06, 0.47, 0.61]","[1.1, 1.8, 0.4]",1.02,0.90,NaN
150,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"[1.18, 1.36, 1.01]","[1.8, 0.4, 0.7]",0.68,0.63,NaN
151,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"[1.65, 1.25, 1.29]","[0.4, 0.7, 0.2]",1.01,0.96,NaN


Средние метрики Holt–Winter's по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,212.06,167.22,35.99
1,АКТЮБИНСКАЯ ОБЛАСТЬ,14.34,11.83,23.21
2,АЛМАТИНСКАЯ ОБЛАСТЬ,80.16,71.08,193.80
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,98.05,89.04,27.16
4,ГШЫМКЕНТ,3.15,2.55,96.04
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,11.06,9.95,41.37
6,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,24.04,20.98,11.99
7,КАРАГАНДИНСКАЯ ОБЛАСТЬ,169.66,150.24,15.53
8,КОСТАНАЙСКАЯ ОБЛАСТЬ,117.46,108.56,29.30
9,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,8.42,5.55,31.71


## SARIMA

In [6]:
results_sarima = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    ts = ts + epsilon
    ts_log = np.log(ts)

    if len(ts_log) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. для авто-ARIMA, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train_log = ts_log[ts_log.index < test_start]
        test_log  = ts_log[(ts_log.index >= test_start) & (ts_log.index <= test_end)]
        if len(train_log) < 12 + horizon or len(test_log) < horizon:
            continue

        # автоподбор на лог-данных
        use_seasonal = len(train_log) >= 2 * 12

        sarima_log = auto_arima(
            train_log,
            seasonal=use_seasonal,
            m=12 if use_seasonal else 1,
            D=1 if use_seasonal else 0,      # фиксируем порядок сезонной разности
            seasonal_test=None,               # пропустить nsdiffs
            boxcox=True,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )
      
        # прогноз в лог-шкале
        fc_log = sarima_log.predict(n_periods=horizon, return_conf_int=False)

        # возвращаем прогноз в исходные единицы
        fc = np.exp(fc_log) - epsilon
        actual = np.exp(test_log.values) - epsilon  # но exp(log(x)) == x

        # метрики на исходном уровне
        rmse = np.sqrt(mean_squared_error(actual, fc))
        mae  = mean_absolute_error(actual, fc)
        mape = (np.abs((actual - fc) / actual).mean()) * 100

        results_sarima.append({
            "Регион":         region,
            "Test start":     test_start.strftime("%Y-%m"),
            "Test end":       test_end.strftime("%Y-%m"),
            "order":          sarima_log.order,
            "seasonal_order": sarima_log.seasonal_order,
            "RMSE":           round(rmse,2),
            "MAE":            round(mae,2),
            "MAPE_%":         round(mape,2),
            "Forecast":       [round(x,2) for x in fc],
            "Actual":         [round(y,2) for y in actual]
        })
# формируем DataFrame с результатами
res_sarima = pd.DataFrame(results_sarima)
res_sarima.to_excel("results/Свиньи - Результаты прогнозов SARIMA v2.xlsx", index=False)
print("Результаты прогнозов SARIMA на 3 месяца:")

display(res_sarima)

final_sarima = (
    res_sarima
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_sarima.to_excel("results/Свиньи - Результаты прогнозов SARIMA средние v2.xlsx", index=False)
print("Средние метрики SARIMA по регионам (rolling-3):")
display(final_sarima)

Результаты прогнозов SARIMA на 3 месяца:


,Регион,Test start,Test end,order,seasonal_order,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"(0, 1, 1)","(0, 1, 1, 12)",73.08,61.25,11.18,"[436.41, 525.28, 522.1]","[553.23, 566.96, 496.85]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"(0, 1, 1)","(0, 1, 1, 12)",154.51,110.54,22.52,"[569.74, 567.61, 746.67]","[566.96, 496.85, 488.59]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"(0, 1, 1)","(0, 1, 1, 12)",427.94,339.58,70.26,"[567.02, 745.43, 1171.78]","[496.85, 488.59, 480.05]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"(0, 1, 1)","(0, 1, 1, 12)",390.28,297.82,63.53,"[714.92, 1116.26, 294.65]","[488.59, 480.05, 263.72]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"(0, 1, 1)","(0, 1, 1, 12)",285.98,168.29,35.52,"[975.33, 257.0, 343.66]","[480.05, 263.72, 340.79]"
...,...,...,...,...,...,...,...,...,...,...
148,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"(0, 1, 1)","(1, 1, 1, 12)",1.90,1.28,285.47,"[3.67, 0.9, 1.43]","[0.4, 1.1, 1.8]"
149,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"(3, 1, 0)","(0, 1, 1, 12)",0.75,0.62,112.54,"[0.36, 1.83, 1.47]","[1.1, 1.8, 0.4]"
150,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"(0, 1, 1)","(1, 1, 1, 12)",0.61,0.49,65.70,"[0.92, 0.98, 0.68]","[1.8, 0.4, 0.7]"
151,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"(1, 1, 0)","(1, 1, 1, 12)",1.10,1.01,305.90,"[2.01, 1.24, 1.08]","[0.4, 0.7, 0.2]"


Средние метрики SARIMA по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,199.53,157.56,34.23
1,АКТЮБИНСКАЯ ОБЛАСТЬ,10.68,9.25,17.36
2,АЛМАТИНСКАЯ ОБЛАСТЬ,77.30,67.11,196.48
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,133.94,119.29,34.98
4,ГШЫМКЕНТ,0.76,0.62,38.45
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,8.15,6.87,24.22
6,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,23.06,20.13,11.33
7,КАРАГАНДИНСКАЯ ОБЛАСТЬ,175.48,154.15,16.57
8,КОСТАНАЙСКАЯ ОБЛАСТЬ,112.30,102.88,25.73
9,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,5.85,4.02,28.63


## Facebook Prophet

In [7]:
results_prophet = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    if len(ts) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. данных, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]
        if len(train) < 12 + horizon or len(test) < horizon:
            continue

        # Подготовка данных для Prophet
#         df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
# # подготовка для одного региона
        df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
        df_prophet["y"] = np.log(df_prophet["y"] + epsilon)

        m = Prophet()
        m.fit(df_prophet)

        future = m.make_future_dataframe(periods=horizon, freq="MS")
        forecast = m.predict(future)

        # берем только прогнозные точки
        yhat_log = forecast["yhat"].values[-horizon:]
        fc = np.exp(yhat_log) - epsilon

        # m = Prophet()
        # m.fit(df_prophet)

        # # Создаем DataFrame будущих дат и делаем прогноз
        # # future = m.make_future_dataframe(periods=horizon, freq="MS")
        # # forecast = m.predict(future)

        # # Отбираем только наши горизонты
        # fc = forecast.set_index("ds")["yhat"].loc[test.index].values
        actual = test.values

        # Расчет метрик
        rmse  = np.sqrt(mean_squared_error(actual, fc))
        mae   = mean_absolute_error(actual, fc)
        mape  = (np.abs((actual - fc) / actual).mean()) * 100

        results_prophet.append({
            "Регион":     region,
            "Test start": test_start.strftime("%Y-%m"),
            "Test end":   test_end.strftime("%Y-%m"),
            "RMSE":       round(rmse, 2),
            "MAE":        round(mae, 2),
            "MAPE_%":     round(mape, 2),
            "Forecast":   [round(x, 2) for x in fc],
            "Actual":     [round(x, 2) for x in actual]
        })

# Собираем результаты в DataFrame
res_prophet = pd.DataFrame(results_prophet)
res_prophet.to_excel("results/Свиньи - Результаты прогнозов Prophet v2.xlsx", index=False)
print("Результаты прогнозов Prophet на 3 месяца:")
display(res_prophet)

final_prophet = (
    res_prophet
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_prophet.to_excel("results/Свиньи - Результаты прогнозов Prophet средние v2.xlsx", index=False)
print("Средние метрики Prophet по регионам (rolling-3):")
display(final_prophet)


12:07:07 - cmdstanpy - INFO - Chain [1] start processing
12:07:08 - cmdstanpy - INFO - Chain [1] done processing
12:07:08 - cmdstanpy - INFO - Chain [1] start processing
12:07:08 - cmdstanpy - INFO - Chain [1] done processing
12:07:08 - cmdstanpy - INFO - Chain [1] start processing
12:07:08 - cmdstanpy - INFO - Chain [1] done processing
12:07:09 - cmdstanpy - INFO - Chain [1] start processing
12:07:09 - cmdstanpy - INFO - Chain [1] done processing
12:07:09 - cmdstanpy - INFO - Chain [1] start processing
12:07:09 - cmdstanpy - INFO - Chain [1] done processing
12:07:09 - cmdstanpy - INFO - Chain [1] start processing
12:07:10 - cmdstanpy - INFO - Chain [1] done processing
12:07:10 - cmdstanpy - INFO - Chain [1] start processing
12:07:10 - cmdstanpy - INFO - Chain [1] done processing
12:07:10 - cmdstanpy - INFO - Chain [1] start processing
12:07:10 - cmdstanpy - INFO - Chain [1] done processing
12:07:11 - cmdstanpy - INFO - Chain [1] start processing
12:07:11 - cmdstanpy - INFO - Chain [1]

Результаты прогнозов Prophet на 3 месяца:


,Регион,Test start,Test end,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,63.75,56.52,10.84,"[506.29, 540.82, 593.32]","[553.23, 566.96, 496.85]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,154.64,123.33,24.94,"[547.69, 601.94, 734.21]","[566.96, 496.85, 488.59]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,442.12,352.84,72.98,"[596.11, 726.91, 1201.0]","[496.85, 488.59, 480.05]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,418.36,320.44,69.49,"[708.9, 1168.36, 316.42]","[488.59, 480.05, 263.72]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,364.46,222.68,48.43,"[1110.38, 297.84, 344.38]","[480.05, 263.72, 340.79]"
...,...,...,...,...,...,...,...,...
148,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,0.64,0.53,43.57,"[0.54, 0.67, 0.79]","[0.4, 1.1, 1.8]"
149,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,0.67,0.55,44.36,"[0.64, 0.75, 0.53]","[1.1, 1.8, 0.4]"
150,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,0.60,0.44,38.94,"[0.79, 0.56, 0.56]","[1.8, 0.4, 0.7]"
151,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,0.47,0.36,150.64,"[0.61, 0.61, 0.97]","[0.4, 0.7, 0.2]"


Средние метрики Prophet по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,190.30,147.15,31.66
1,АКТЮБИНСКАЯ ОБЛАСТЬ,51.28,42.91,59.60
2,АЛМАТИНСКАЯ ОБЛАСТЬ,97.30,89.04,302.86
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,75.66,68.49,19.05
4,ГШЫМКЕНТ,4.07,3.21,58.29
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,15.72,13.11,40.19
6,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,27.04,24.13,13.42
7,КАРАГАНДИНСКАЯ ОБЛАСТЬ,170.84,152.82,15.19
8,КОСТАНАЙСКАЯ ОБЛАСТЬ,124.58,109.19,33.40
9,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,12.52,8.36,62.14


In [8]:
# Переименуем колонки с MAPE, чтобы было понятно, к какому методу относятся
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW"})
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA"})
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet"})

# Мёрджим по региону
summary = (
    hw[["Регион", "MAPE_HW"]]
    .merge(sar[["Регион", "MAPE_SARIMA"]], on="Регион")
    .merge(pr[["Регион", "MAPE_Prophet"]], on="Регион")
)

# Определяем для каждой строки, какой столбец MAPE минимален
# idxmin вернёт название столбца с минимальным значением
summary["Best_method"] = summary[["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]] \
                           .idxmin(axis=1) \
                           .str.replace("MAPE_","")  # убираем префикс для красоты

# Если нужно, можно сразу отсортировать
# summary = summary.sort_values("Best_method")

# допустим, у вас уже есть summary
summary = summary.round({
    "MAPE_HW": 2,
    "MAPE_SARIMA": 2,
    "MAPE_Prophet": 2
})

# Готово!
print(summary.to_string(index=False))
summary.to_excel("results/Свиньи - Лучшие модели v2.xlsx", index=False)


                        Регион  MAPE_HW  MAPE_SARIMA  MAPE_Prophet Best_method
           АКМОЛИНСКАЯ ОБЛАСТЬ    35.99        34.23         31.66     Prophet
           АКТЮБИНСКАЯ ОБЛАСТЬ    23.21        17.36         59.60      SARIMA
           АЛМАТИНСКАЯ ОБЛАСТЬ   193.80       196.48        302.86          HW
ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ    27.16        34.98         19.05     Prophet
                      ГШЫМКЕНТ    96.04        38.45         58.29      SARIMA
            ЖАМБЫЛСКАЯ ОБЛАСТЬ    41.37        24.22         40.19      SARIMA
 ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ    11.99        11.33         13.42      SARIMA
        КАРАГАНДИНСКАЯ ОБЛАСТЬ    15.53        16.57         15.19     Prophet
          КОСТАНАЙСКАЯ ОБЛАСТЬ    29.30        25.73         33.40      SARIMA
        КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ    31.71        28.63         62.14      SARIMA
                  ОБЛАСТЬ АБАЙ    34.83        38.66         75.97          HW
                ОБЛАСТЬ ЖЕТІСУ    58.23       168.03

In [9]:
#FOLDER = Path("results")  # папка, где лежат файлы
FILE_HW      = "results/Свиньи - Результаты прогнозов ХВ средние v2.xlsx"
FILE_SARIMA  = "results/Свиньи - Результаты прогнозов SARIMA средние v2.xlsx"
FILE_PROPHET = "results/Свиньи - Результаты прогнозов Prophet средние v2.xlsx"


# === Загрузка исходных таблиц ===
final_hw      = pd.read_excel(FILE_HW)
final_sarima  = pd.read_excel(FILE_SARIMA)
final_prophet = pd.read_excel(FILE_PROPHET)

# Ожидаемые столбцы: 'Регион', 'MAPE_%', 'MAE' (и/или 'RMSE')
# Переименуем для прозрачности
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW", "MAE": "MAE_HW"})[["Регион","MAPE_HW","MAE_HW"]]
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA", "MAE": "MAE_SARIMA"})[["Регион","MAPE_SARIMA","MAE_SARIMA"]]
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet", "MAE": "MAE_Prophet"})[["Регион","MAPE_Prophet","MAE_Prophet"]]

# === Объединяем по региону ===
summary = (
    hw.merge(sar, on="Регион", how="inner")
      .merge(pr,  on="Регион", how="inner")
)


In [10]:
THRESHOLD_MAPE = 1000.0  # порог

mape_cols = ["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]
mae_cols  = ["MAE_HW","MAE_SARIMA","MAE_Prophet"]

# 1) Приведём метрики к числам (на всякий случай ещё раз)
for c in mape_cols + mae_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

def choose_best_simple(row):
    # Берём числовые серии и подменяем NaN на +inf, чтобы .idxmin() стабильно работал
    mape_s = row[mape_cols].astype(float).fillna(np.inf)
    mae_s  = row[mae_cols].astype(float).fillna(np.inf)

    min_mape = mape_s.min()

    # Если все MAPE были NaN -> min = +inf
    if np.isinf(min_mape):
        criterion = "MAE"
        winner_col = mae_s.idxmin()
    elif min_mape <= THRESHOLD_MAPE:
        criterion = "MAPE"
        winner_col = mape_s.idxmin()
    else:
        criterion = "MAE"
        winner_col = mae_s.idxmin()

    method = winner_col.split("_")[-1]  # HW / SARIMA / Prophet

    return pd.Series({
        "Best_method": method,
        "Best_criterion": criterion,
        "Best_MAPE": float(mape_s.replace(np.inf, np.nan).min()),
        "Best_MAE": float(mae_s.replace(np.inf, np.nan).min())
    })

best = summary.apply(choose_best_simple, axis=1)

result = pd.concat([summary, best], axis=1)

# Округление и сохранение
for c in mape_cols + mae_cols + ["Best_MAPE","Best_MAE"]:
    result[c] = result[c].round(2)

# Если у вас есть переменная OUT_FILE — используйте её. Иначе:
OUT_FILE = "results/Свиньи - Лучшие модели (MAPE_then_MAE) v2.xlsx"
result.sort_values(["Best_method","Регион"]).to_excel(OUT_FILE, index=False)

result

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,35.99,167.22,34.23,157.56,31.66,147.15,Prophet,MAPE,31.66,147.15
1,АКТЮБИНСКАЯ ОБЛАСТЬ,23.21,11.83,17.36,9.25,59.60,42.91,SARIMA,MAPE,17.36,9.25
2,АЛМАТИНСКАЯ ОБЛАСТЬ,193.80,71.08,196.48,67.11,302.86,89.04,HW,MAPE,193.80,67.11
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,27.16,89.04,34.98,119.29,19.05,68.49,Prophet,MAPE,19.05,68.49
4,ГШЫМКЕНТ,96.04,2.55,38.45,0.62,58.29,3.21,SARIMA,MAPE,38.45,0.62
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,41.37,9.95,24.22,6.87,40.19,13.11,SARIMA,MAPE,24.22,6.87
6,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,11.99,20.98,11.33,20.13,13.42,24.13,SARIMA,MAPE,11.33,20.13
7,КАРАГАНДИНСКАЯ ОБЛАСТЬ,15.53,150.24,16.57,154.15,15.19,152.82,Prophet,MAPE,15.19,150.24
8,КОСТАНАЙСКАЯ ОБЛАСТЬ,29.30,108.56,25.73,102.88,33.40,109.19,SARIMA,MAPE,25.73,102.88
9,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,31.71,5.55,28.63,4.02,62.14,8.36,SARIMA,MAPE,28.63,4.02
